# 06. Image Embedding — CLIP 기반 상품 임베딩 생성

- **목표**: 상품 이미지 → 512D 벡터 임베딩 생성 → Supabase pgvector 저장
- **모델**: OpenAI CLIP (ViT-B/32) — 이미지+텍스트 사전학습, 패션 도메인 우수
- **파이프라인**: 상품 이미지 URL → CLIP → 512D 벡터 → pgvector
- **활용**: 유저 업로드 이미지 → 임베딩 → 가장 유사한 상품 N개 검색

In [ ]:
# 패키지 설치 (최초 1회)
# !pip install open-clip-torch pillow requests supabase tqdm

In [ ]:
import open_clip
import torch
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import json
import os
from tqdm import tqdm
import time

# ── Config ──
EMBEDDING_DIM = 512
CLIP_MODEL = 'ViT-B-32'
CLIP_PRETRAINED = 'openai'
BATCH_SIZE = 16
DATA_DIR = os.path.expanduser('~/Documents/personal_project/codi/model/data')
SAVE_DIR = os.path.expanduser('~/Documents/personal_project/codi/model/saved_model')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'OpenCLIP: {open_clip.__version__}')
print(f'Device: {"mps" if torch.backends.mps.is_available() else "cpu"}')

## 1. CLIP 모델 로드

CLIP (Contrastive Language-Image Pre-training)을 선택한 이유:
- 4억 개 이미지-텍스트 쌍으로 사전학습 → 패션 이미지 이해 우수
- ViT-B/32 출력이 **512D** → pgvector(512) 스키마와 정확히 일치
- Triplet Loss 커스텀 학습 불필요, 제로샷으로 바로 사용 가능
- 향후 "검정 가죽 부츠" 같은 텍스트 검색도 같은 임베딩 공간에서 가능

In [ ]:
# CLIP 모델 + 전처리 로드
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
model = model.to(device)
model.eval()

print(f'CLIP {CLIP_MODEL} loaded on {device}')
print(f'Image input size: {preprocess.transforms[0].size}')

# 임베딩 차원 확인
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224).to(device)
    dummy_emb = model.encode_image(dummy)
    print(f'Embedding dimension: {dummy_emb.shape[1]} (expected {EMBEDDING_DIM})')

In [ ]:
from supabase import create_client

# Supabase 연결
SUPABASE_URL = 'https://REMOVED_PROJECT_ID.supabase.co'
SUPABASE_KEY = '***REMOVED_SUPABASE_ANON_JWT***'

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# 모든 상품 조회 (임베딩이 없는 것만)
response = supabase.table('products').select('id, name, brand, category, image_url').is_('embedding', 'null').execute()
products = response.data
print(f'임베딩 미생성 상품: {len(products)}개')

# 샘플 확인
if products:
    print(f'\n예시: {products[0]["name"]} ({products[0]["brand"]}) - {products[0]["category"]}')
    print(f'이미지: {products[0]["image_url"][:80]}...')

## 3. 이미지 다운로드 + 임베딩 생성

In [ ]:
def download_image(url, timeout=10):
    """URL에서 이미지를 다운로드하여 PIL Image로 반환"""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
        }
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        return img
    except Exception as e:
        return None


def generate_embedding(image, model, preprocess, device):
    """PIL Image → CLIP 512D 임베딩 벡터"""
    img_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model.encode_image(img_tensor)
        # L2 정규화 (코사인 유사도 사용을 위해)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy().flatten()


# 테스트: 첫 번째 상품 이미지로 임베딩 생성
if products:
    test_img = download_image(products[0]['image_url'])
    if test_img:
        test_emb = generate_embedding(test_img, model, preprocess, device)
        print(f'테스트 임베딩 shape: {test_emb.shape}')
        print(f'L2 norm: {np.linalg.norm(test_emb):.4f} (정규화 후 ≈1.0)')
        print(f'벡터 샘플: [{test_emb[0]:.4f}, {test_emb[1]:.4f}, ..., {test_emb[-1]:.4f}]')
    else:
        print('테스트 이미지 다운로드 실패')

In [ ]:
# 전체 상품 임베딩 생성
embeddings = {}  # product_id → embedding vector
failed = []      # 실패한 상품 목록

print(f'총 {len(products)}개 상품 임베딩 생성 시작...\n')

for i, product in enumerate(tqdm(products, desc='Generating embeddings')):
    pid = product['id']
    url = product['image_url']
    
    # 이미지 다운로드
    img = download_image(url)
    if img is None:
        failed.append({'id': pid, 'name': product['name'], 'url': url})
        continue
    
    # 임베딩 생성
    emb = generate_embedding(img, model, preprocess, device)
    embeddings[pid] = emb
    
    # 과도한 요청 방지
    if (i + 1) % 50 == 0:
        time.sleep(1)

print(f'\n완료: {len(embeddings)}개 성공, {len(failed)}개 실패')
if failed:
    print('\n실패 목록:')
    for f in failed[:10]:
        print(f'  - {f["name"]}: {f["url"][:60]}...')

## 4. 임베딩 품질 검증

같은 카테고리의 상품끼리 임베딩이 가까운지 확인

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 카테고리별 상품 그룹핑
cat_products = {}
for p in products:
    if p['id'] in embeddings:
        cat = p['category']
        if cat not in cat_products:
            cat_products[cat] = []
        cat_products[cat].append(p['id'])

# 카테고리 내 평균 유사도 vs 카테고리 간 평균 유사도
all_ids = list(embeddings.keys())
all_embs = np.array([embeddings[pid] for pid in all_ids])
sim_matrix = cosine_similarity(all_embs)

id_to_idx = {pid: i for i, pid in enumerate(all_ids)}

print('카테고리 내 평균 유사도 (높을수록 좋음):')
intra_sims = []
for cat, pids in sorted(cat_products.items()):
    if len(pids) < 2:
        continue
    idxs = [id_to_idx[pid] for pid in pids if pid in id_to_idx]
    cat_sim = sim_matrix[np.ix_(idxs, idxs)]
    # 대각선 제외 평균
    mask = ~np.eye(len(idxs), dtype=bool)
    avg_sim = cat_sim[mask].mean()
    intra_sims.append(avg_sim)
    print(f'  {cat:12s}: {avg_sim:.4f} ({len(pids)} items)')

# 전체 평균 (카테고리 무관)
mask = ~np.eye(len(all_ids), dtype=bool)
inter_sim = sim_matrix[mask].mean()
print(f'\n카테고리 내 평균: {np.mean(intra_sims):.4f}')
print(f'전체 평균 (무관): {inter_sim:.4f}')
print(f'\n→ 카테고리 내 유사도가 전체 평균보다 높으면 임베딩 품질 양호')

In [ ]:
# 유사 상품 검색 테스트: 임의의 상품과 가장 유사한 5개
import random

test_product = random.choice(products)
while test_product['id'] not in embeddings:
    test_product = random.choice(products)

test_idx = id_to_idx[test_product['id']]
sims = sim_matrix[test_idx]
top_indices = np.argsort(sims)[::-1][1:6]  # 자기 자신 제외 top 5

print(f'Query: {test_product["name"]} ({test_product["brand"]}, {test_product["category"]})')
print(f'\nTop 5 유사 상품:')
for rank, idx in enumerate(top_indices, 1):
    pid = all_ids[idx]
    p = next(p for p in products if p['id'] == pid)
    print(f'  {rank}. [{sims[idx]:.4f}] {p["name"]} ({p["brand"]}, {p["category"]})')

## 5. Supabase pgvector에 임베딩 업로드

In [ ]:
# pgvector에 임베딩 저장
# Supabase의 vector 컬럼은 '[0.1, 0.2, ...]' 형식의 문자열로 저장

success_count = 0
error_count = 0

print(f'총 {len(embeddings)}개 임베딩 업로드 시작...\n')

for pid, emb in tqdm(embeddings.items(), desc='Uploading to pgvector'):
    try:
        # numpy array → Python list (JSON 직렬화 가능)
        emb_list = emb.tolist()
        
        supabase.table('products').update(
            {'embedding': emb_list}
        ).eq('id', pid).execute()
        
        success_count += 1
    except Exception as e:
        error_count += 1
        if error_count <= 3:
            print(f'  Error [{pid}]: {e}')

print(f'\n업로드 완료: {success_count}개 성공, {error_count}개 실패')

## 6. pgvector 유사도 검색 테스트

Supabase SQL 함수를 만들어 벡터 유사도 검색 수행

In [ ]:
# pgvector 유사도 검색 SQL 함수 (Supabase Dashboard에서 실행)
create_function_sql = """
-- pgvector 확장 활성화 (이미 되어있을 수 있음)
CREATE EXTENSION IF NOT EXISTS vector;

-- 유사 상품 검색 함수
CREATE OR REPLACE FUNCTION match_products(
  query_embedding vector(512),
  match_threshold float DEFAULT 0.5,
  match_count int DEFAULT 10,
  filter_category text DEFAULT NULL
)
RETURNS TABLE (
  id uuid,
  name text,
  brand text,
  category text,
  color text,
  style text,
  price numeric,
  image_url text,
  affiliate_url text,
  similarity float
)
LANGUAGE plpgsql
AS $$
BEGIN
  RETURN QUERY
  SELECT
    p.id, p.name, p.brand, p.category, p.color, p.style,
    p.price, p.image_url, p.affiliate_url,
    1 - (p.embedding <=> query_embedding) AS similarity
  FROM products p
  WHERE p.embedding IS NOT NULL
    AND 1 - (p.embedding <=> query_embedding) > match_threshold
    AND (filter_category IS NULL OR p.category = filter_category)
  ORDER BY p.embedding <=> query_embedding
  LIMIT match_count;
END;
$$;
"""

print('아래 SQL을 Supabase Dashboard SQL Editor에서 실행하세요:')
print('=' * 60)
print(create_function_sql)

In [ ]:
# RPC로 유사도 검색 테스트
test_product = random.choice(products)
while test_product['id'] not in embeddings:
    test_product = random.choice(products)

test_emb = embeddings[test_product['id']].tolist()

print(f'Query: {test_product["name"]} ({test_product["brand"]}, {test_product["category"]})')
print(f'\npgvector 유사도 검색 결과:')

try:
    result = supabase.rpc('match_products', {
        'query_embedding': test_emb,
        'match_threshold': 0.3,
        'match_count': 5
    }).execute()
    
    for i, item in enumerate(result.data, 1):
        print(f'  {i}. [{item["similarity"]:.4f}] {item["name"]} ({item["brand"]}, {item["category"]}) ${item["price"]}')
except Exception as e:
    print(f'RPC 호출 실패: {e}')
    print('→ Supabase Dashboard에서 match_products 함수를 먼저 생성하세요')

## 7. 임베딩 로컬 백업 저장

In [ ]:
# 임베딩을 로컬에 저장 (재실행 없이 업로드만 하고 싶을 때 사용)
save_path = os.path.join(SAVE_DIR, 'product_embeddings.npz')

ids = list(embeddings.keys())
vecs = np.array([embeddings[pid] for pid in ids])

np.savez(save_path, ids=np.array(ids), embeddings=vecs)
print(f'저장 완료: {save_path}')
print(f'  상품 수: {len(ids)}')
print(f'  벡터 shape: {vecs.shape}')
print(f'  파일 크기: {os.path.getsize(save_path) / 1024:.1f} KB')

## 요약

### 파이프라인
1. **상품 카탈로그**: 이미지 URL → CLIP → 512D 벡터 → Supabase pgvector (배치, 이 노트북)
2. **유저 업로드**: 사진 → Edge Function에서 CLIP 추론 → pgvector 검색 → 유사 상품 반환

### 다음 단계
- [ ] Supabase Edge Function으로 서버사이드 임베딩 생성 (유저 업로드용)
- [ ] Flutter 앱에서 `match_products` RPC 호출 연동
- [ ] 기존 메타데이터 기반 유사 검색 → 임베딩 유사도로 전환/보완